In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Leemos los datos
df = pd.read_csv("data.csv")

# Separamos inputs y target
X = df.drop(['pl_name', 'pl_eqt'], axis=1).values
y = df['pl_eqt'].values

# Creamos tests de entrenamiento y validación
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2
)

# Escalamos las feautres para que la red se alimente más fácil
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [28]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Creamos el modelo
model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)   # la última capa tiene activación lineal porque queremos datos de 0 a infty
])

# Le decimos al modelo qué hacer
model.compile(
    optimizer='adam',
    loss='mse',                # mean squared error
    metrics=['mae']            # mean absolute error (more interpretable)
)

In [38]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Si en el decenso del gradiente se nos va por trochas que no son, entonces volvemos
# a donde estábamos bien
early_stop = EarlyStopping(
    monitor='val_loss', patience=20, restore_best_weights=True
)

# Guardamos checkpoints del modelo que mejor se comporan
checkpoint = ModelCheckpoint(
    'best_model.keras', monitor='val_loss', save_best_only=True
)

# Entrenamos
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,          
    epochs=500,
    batch_size=128,
    callbacks=[early_stop, checkpoint],
    verbose=0
)

In [39]:
# Evaluamos el modelo en el test set
test_loss, test_mae = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Test MAE: {test_mae:.2f}")   # Imprimios el Mean Average Error

#  Guardamos el modelo para luego
model.save('model.keras')   # or .h5 for older versions

Test MAE: 5.94


In [64]:
# Cargamos el modelo
loaded_model = tf.keras.models.load_model('model.keras')

# Función para hallar los datos de un planeta en específico
def get_data(pl_name, which = 0):

    # Hallamos los inputs
    row = df.loc[df["pl_name"] == pl_name]
    inputs = row.drop(['pl_name', 'pl_eqt'], axis=1).values[which]
    temp = row['pl_eqt'].values[which]

    print("Planetas posibles:")
    print(row)

    return inputs.reshape(1, -1), temp
    
# Cargamos los datos
new_data, new_temp = get_data("HAT-P-63 b")
new_data_scaled = scaler.transform(new_data)

# Predecimos
print("\nPREDICIENDO ")
predictions = loaded_model.predict(new_data_scaled)

print("\nResultados ")
print(f"Temperatura reportada del modelo: {new_temp}")
print(f"Predicción del modelo: {predictions[0][0]}")
print(f"Error relativo: {(100.0*(1 - (new_temp/predictions[0][0]))):.2f}%")

Planetas posibles:
         pl_name  pl_orbper  pl_orbsmax    pl_rade  pl_insol  st_teff  pl_eqt  \
5419  HAT-P-63 b   3.377728     0.04294  12.542849     387.4   5400.0  1237.0   

        pl_tranmid  st_rad  st_mass  
5419  2.456383e+06  0.9661    0.925  

PREDICIENDO 
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step

Resultados 
Temperatura reportada del modelo: 1237.0
Predicción del modelo: 1213.57373046875
Error relativo: -1.93%
